# 📦 4. Final Dataset Creation & Integration

Este notebook consolida la información estructural (CSV) con las representaciones semánticas (Embeddings `.npy`) para generar objetos `Dataset` de PyTorch optimizados, listos para consumo en el módulo de entrenamiento supervisado.

---

## 📋 Descripción general

El script carga por separado los metadatos tabulares (`.csv`) y los tensores numéricos (`.npy`), verifica y garantiza la alineación estricta de índices mediante IDs únicos, y encapsula toda la información en una clase `torch.utils.data.Dataset` personalizada.

Cada muestra del dataset contiene:
- `X`: embeddings por residuo, con forma `(100, 768)`.
- `G`: embedding global/contextual, con forma `(768,)`.
- `M`: máscara binaria de padding, con forma `(100,)`.
- `seq`: secuencia de aminoácidos cruda (para análisis posterior).
- `y`: etiqueta de clase, con forma `(1,)`.

---

## 🛠️ Funcionalidades principales

1. **Carga de datos multiformato**
   - Importa los archivos CSV con identificadores, secuencias crudas y etiquetas de clase.
   - Importa los tensores `.npy` con embeddings por residuo (`X`), embedding global (`G`), máscara (`M`) y etiquetas (`y`).

2. **Verificación de alineación**
   - Verifica que los IDs de los metadatos y los tensores estén en el mismo orden.
   - Evita desajustes label-embedding que podrían causar fugas de datos o errores de entrenamiento.

3. **Encapsulación en `AMPDataset`**
   - Define una clase optimizada que retorna por índice un diccionario estandarizado con `X`, `G`, `M`, `seq` y `y`.
   - Convierte los arrays NumPy a tensores PyTorch de tipo `float`.

4. **Serialización eficiente**
   - Guarda los datasets como archivos `.pt` nativos de PyTorch.
   - Preserva tensores, estructura y metadatos para carga instantánea durante `training.ipynb`.

## 📦 Importación de dependencias

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset

## ⚙️ Configuración de rutas

In [2]:
INPUT_EMB = "/kaggle/input/datasets/user/new-embeddings"
INPUT_CSV = "/kaggle/input/datasets/user/new-data-consolidation"
OUTPUT_DATASETS = "/kaggle/working/"

os.makedirs(OUTPUT_DATASETS, exist_ok=True)

## 📂 Carga de metadatos y tensores numéricos

In [3]:
def load_metadata(split):
    """Carga IDs, secuencias crudas y etiquetas desde CSV."""
    path = os.path.join(INPUT_CSV, f"{split}.csv")
    df = pd.read_csv(path)
    return df[['id', 'sequence', 'label']].copy()

def load_tensors(split):
    """Carga tensores numpy precomputados (X, G, M, y) e IDs."""
    base = INPUT_EMB
    X = np.load(os.path.join(base, f"{split}_embeddings.npy")) # (N, 100, 768)
    G = np.load(os.path.join(base, f"{split}_global.npy"))    # (N, 768) - CONTEXTO GLOBAL
    M = np.load(os.path.join(base, f"{split}_masks.npy"))     # (N, 100)
    y = np.load(os.path.join(base, f"{split}_labels.npy"))    # (N,)
    
    # Cargar IDs
    ids_df = pd.read_csv(os.path.join(base, f"{split}_ids.csv"))
    ids = ids_df['id'].tolist() if 'id' in ids_df.columns else ids_df.iloc[:, 0].tolist()
    
    return X, G, M, y, ids

# Carga masiva
print("⏳ Cargando datos en memoria...")
train_meta = load_metadata("train")
val_meta   = load_metadata("val")
test_meta  = load_metadata("test")

train_X, train_G, train_M, train_y, train_ids = load_tensors("train")
val_X,   val_G,   val_M,   val_y,   val_ids   = load_tensors("val")
test_X,  test_G,  test_M,  test_y,  test_ids  = load_tensors("test")

print("✅ Carga completada.")

⏳ Cargando datos en memoria...
✅ Carga completada.


## 🔍 Verificación de Alineación

In [4]:
def verify_alignment(meta_df, tensor_ids, split_name):
    """Verifica que los IDs estén en el mismo orden."""
    meta_ids = meta_df['id'].tolist()
    if meta_ids == tensor_ids:
        print(f"✅ {split_name.upper()}: Alineación perfecta ({len(meta_ids)} registros)")
        return 
    else:
        raise ValueError(f"❌ CRITICAL ERROR: ID mismatch in {split_name}. Data leakage risk!")

# Ejecutar verificación
verify_alignment(train_meta, train_ids, "train")
verify_alignment(val_meta, val_ids, "val")
verify_alignment(test_meta, test_ids, "test")

✅ TRAIN: Alineación perfecta (13766 registros)
✅ VAL: Alineación perfecta (4128 registros)
✅ TEST: Alineación perfecta (15685 registros)


## 🧩 Clase `AMPDataset` (Custom PyTorch Dataset)

In [5]:
class AMPDataset(Dataset):
    """
    Dataset para clasificador AMP con Gated Pooling / Attention.
    
    Returns:
        X:   (100, 768)  - Embeddings por residuo
        G:   (768,)      - Embedding global (CLS/Mean)
        M:   (100,)      - Máscara de atención
        seq: str         - Secuencia original (para análisis posterior)
        y:   (1,)        - Etiqueta
    """
    def __init__(self, X, G, M, sequences, labels):
        self.X = torch.from_numpy(X).float()
        self.G = torch.from_numpy(G).float()
        self.M = torch.from_numpy(M).float()
        self.sequences = sequences
        self.y = torch.from_numpy(labels).float().unsqueeze(1) # Shape (N, 1)

        assert len(self.X) == len(self.G) == len(self.M) == len(self.y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],
            "G": self.G[idx],
            "M": self.M[idx],
            "seq": self.sequences[idx],
            "y": self.y[idx]
        }

## 💾 Serialización Persistente

In [6]:
# Instanciar datasets
train_dataset = AMPDataset(train_X, train_G, train_M, train_meta['sequence'].tolist(), train_y)
val_dataset   = AMPDataset(val_X, val_G, val_M, val_meta['sequence'].tolist(), val_y)
test_dataset  = AMPDataset(test_X, test_G, test_M, test_meta['sequence'].tolist(), test_y)

print(f"📦 Datasets creados:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val:   {len(val_dataset)} samples")
print(f"   Test:  {len(test_dataset)} samples")

# Guardar en Drive
torch.save(train_dataset, os.path.join(OUTPUT_DATASETS, "train_dataset.pt"))
torch.save(val_dataset,   os.path.join(OUTPUT_DATASETS, "val_dataset.pt"))
torch.save(test_dataset,  os.path.join(OUTPUT_DATASETS, "test_dataset.pt"))

print(f"\n💾 Guardado exitoso en: {OUTPUT_DATASETS}")

📦 Datasets creados:
   Train: 13766 samples
   Val:   4128 samples
   Test:  15685 samples

💾 Guardado exitoso en: /kaggle/working/


## 👁️ Validación de un sample

In [7]:
sample = train_dataset[0]
print(f"\n🔎 Sample [0]:")
print(f"   X shape (Residuos): {sample['X'].shape}  (dtype: {sample['X'].dtype})")
print(f"   G shape (Global):   {sample['G'].shape}  (dtype: {sample['G'].dtype})")
print(f"   M shape (Mask):     {sample['M'].shape}  (sum: {sample['M'].sum().item():.0f} reales)")
print(f"   Seq:                {sample['seq'][:30]}... (len={len(sample['seq'])})")
print(f"   Label:              {sample['y'].item():.0f}")


🔎 Sample [0]:
   X shape (Residuos): torch.Size([100, 768])  (dtype: torch.float32)
   G shape (Global):   torch.Size([768])  (dtype: torch.float32)
   M shape (Mask):     torch.Size([100])  (sum: 12 reales)
   Seq:                GMSYERLLCLVL... (len=12)
   Label:              0


# 🏁 Resumen del Pipeline de Datasets | ProtFlash

Este notebook cierra la fase de ingesta, vectorización y estructuración, entregando tres objetos `Dataset` serializados y listos para consumo en el módulo de entrenamiento supervisado.

### Entregables

| Archivo | Formato | Volumen | Contenido |
|---|---|---|---|
| `train_dataset.pt` | `AMPDataset` | 13,766 muestras | X, G, M, seq, y |
| `val_dataset.pt` | `AMPDataset` | 4,128 muestras | X, G, M, seq, y |
| `test_dataset.pt` | `AMPDataset` | 15,685 muestras | X, G, M, seq, y |

### Análisis

- **Alineación garantizada:** verificación cruzada de IDs entre CSVs y tensores `.npy` para evitar desajustes label-embedding.
- **Encapsulación PyTorch:** clase `AMPDataset` optimizada para retornar tensores, máscaras, secuencias crudas y labels en un solo dict.
- **Serialización eficiente:** guardado en `.pt` para carga instantánea y sin overhead de parsing en `training.ipynb`.

### Especificaciones del output

- Dimensión del embedding: `(100, 768)` por muestra (ProtFlash per-residue).
- Total de registros sincronizados: 33,589 secuencias validadas.

**Estado final:** los datasets quedan serializados en `/kaggle/working/` como `train_dataset.pt`, `val_dataset.pt` y `test_dataset.pt`, listos para consumo en el módulo de entrenamiento supervisado.